# 01 — Exploratory Data Analysis (EDA)

**Project #8 — Throughput Prediction in a Dense 5G deployment with Transfer Learning**  
Network Data Analysis Laboratory 2025-2026, Politecnico di Milano

---

## Goals
1. Load and inspect both venue datasets (ACC Arena, Salt & Tar)
2. Understand the distribution of all features and the target (throughput)
3. Identify correlations, outliers, and missing values
4. Visualise the spatial layout of users and their throughput
5. Extract insights that guide feature selection and model choice

In [ ]:
import sys
sys.path.insert(0, '..')  # make src/ importable from notebooks/

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_venue, load_all, summary, TRAFFIC_LABELS
from src.utils.visualization import (
    plot_throughput_distribution,
    plot_correlation_heatmap,
    plot_spatial_throughput,
    plot_traffic_type_distribution,
    plot_boxplot_by_traffic,
)

pd.set_option('display.float_format', '{:.4f}'.format)
SAVE_FIGS = True  # set False to skip saving

## 1. Load datasets

In [ ]:
# Load both venues (set nrows=100_000 for quick prototyping)
acc  = load_venue('acc_arena',  data_dir='../data/raw')
salt = load_venue('salt_tar',   data_dir='../data/raw')

print(f"ACC Arena  — shape: {acc.shape}")
print(f"Salt & Tar — shape: {salt.shape}")

In [ ]:
# Quick data overview
acc.head()

## 2. Dataset Summary

In [ ]:
print("=== ACC Arena ===")
display(summary(acc))
print("\n=== Salt & Tar ===")
display(summary(salt))

In [ ]:
# User and RU counts per venue
for name, df in [('ACC Arena', acc), ('Salt & Tar', salt)]:
    print(f"{name}:")
    print(f"  Unique users : {df['user_id'].nunique()}")
    print(f"  Unique RUs   : {df['ru_id'].nunique()}")
    print(f"  Timestamps   : {df['timestamp'].nunique()}")
    print(f"  Samples/user : {len(df) / df['user_id'].nunique():.0f}")
    print()

## 3. Throughput Distribution

In [ ]:
fig = plot_throughput_distribution(acc,  venue='ACC Arena',  save=SAVE_FIGS)
plt.show()
fig = plot_throughput_distribution(salt, venue='Salt & Tar', save=SAVE_FIGS)
plt.show()

In [ ]:
# Side-by-side CDF comparison
fig, ax = plt.subplots(figsize=(10, 5))
for name, df, color in [('ACC Arena', acc, 'steelblue'), ('Salt & Tar', salt, 'coral')]:
    sorted_tp = np.sort(df['throughput'].values)
    cdf = np.arange(1, len(sorted_tp) + 1) / len(sorted_tp)
    ax.plot(sorted_tp, cdf, label=name, color=color)
ax.set_xlabel('Throughput (Mbps)')
ax.set_ylabel('CDF')
ax.set_title('Throughput CDF — ACC Arena vs Salt & Tar')
ax.legend()
plt.show()

## 4. Feature Distributions

In [ ]:
features = ['sinr_dl', 'sinr_ul', 'prb', 'bler']
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for col_idx, feat in enumerate(features):
    for row_idx, (df, label) in enumerate([(acc, 'ACC Arena'), (salt, 'Salt & Tar')]):
        ax = axes[row_idx][col_idx]
        sns.histplot(df[feat], bins=50, kde=True, ax=ax,
                     color='steelblue' if row_idx == 0 else 'coral')
        ax.set_title(f'{feat} — {label}')
        ax.set_xlabel(feat)

plt.suptitle('Feature Distributions (ACC Arena vs Salt & Tar)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
fig = plot_correlation_heatmap(acc,  venue='ACC Arena',  save=SAVE_FIGS)
plt.show()
fig = plot_correlation_heatmap(salt, venue='Salt & Tar', save=SAVE_FIGS)
plt.show()

In [ ]:
# Pairplot (sample for speed)
sample_acc = acc[['sinr_dl', 'sinr_ul', 'prb', 'bler', 'throughput']].sample(5000, random_state=42)
sns.pairplot(sample_acc, diag_kind='kde', plot_kws={'alpha': 0.2, 's': 10})
plt.suptitle('Pairplot — ACC Arena', y=1.01)
plt.show()

## 6. Traffic Type Analysis

In [ ]:
fig = plot_traffic_type_distribution(acc,  venue='ACC Arena',  save=SAVE_FIGS)
plt.show()
fig = plot_traffic_type_distribution(salt, venue='Salt & Tar', save=SAVE_FIGS)
plt.show()

In [ ]:
# Throughput by traffic type
fig = plot_boxplot_by_traffic(acc,  feature='throughput', venue='ACC Arena',  save=SAVE_FIGS)
plt.show()
fig = plot_boxplot_by_traffic(salt, feature='throughput', venue='Salt & Tar', save=SAVE_FIGS)
plt.show()

## 7. Spatial Analysis

In [ ]:
fig = plot_spatial_throughput(acc,  venue='ACC Arena',  save=SAVE_FIGS)
plt.show()
fig = plot_spatial_throughput(salt, venue='Salt & Tar', save=SAVE_FIGS)
plt.show()

In [ ]:
# Mean throughput per RU
ru_stats = acc.groupby('ru_id')['throughput'].agg(['mean', 'std', 'count'])
print('Mean throughput per RU (ACC Arena):')
display(ru_stats.sort_values('mean', ascending=False).head(10))

## 8. Missing Values & Outliers

In [ ]:
print('=== Null counts per column ===')
print('ACC Arena:')
print(acc.isnull().sum())
print('\nSalt & Tar:')
print(salt.isnull().sum())

In [ ]:
# Outlier summary using IQR
def count_outliers(df, col):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    mask = (df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)
    return mask.sum()

for feat in ['sinr_dl', 'sinr_ul', 'prb', 'bler', 'throughput']:
    n_acc  = count_outliers(acc,  feat)
    n_salt = count_outliers(salt, feat)
    print(f"{feat:12s} — ACC Arena: {n_acc:6d}  Salt&Tar: {n_salt:6d}")

## 9. Key Insights

> Fill in after analysing the plots:

- **Throughput**: …
- **SINR**: …
- **BLER**: …
- **PRB**: …
- **Traffic type**: …
- **Spatial patterns**: …
- **Domain shift ACC Arena → Salt&Tar**: …